In [1]:
import pandas as pd 
import numpy as np 
import geopandas as gpd 
# import rioxarray
from pyproj import Transformer
import os
%matplotlib inline
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from os.path import isfile, join
from os import listdir
import time 
import netCDF4 as nc
from netCDF4 import Dataset
import xarray as xr
import rioxarray as rxr


/opt/conda/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('../../data/data_cleaned/patients_FR_IDF_geocoded_adulte_clinique.csv',sep=";")
gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326)).to_crs(epsg=27572)
gdf = gdf[['pseudo_provisoire','geometry']]

### test pour un fichier

In [ ]:
file_path = '../../data/airparif/pollution_chronique/test/horair_IDF_20170106.nc'
import rioxarray as rxr

modis = rxr.open_rasterio(file_path, masked = True)#, band_as_variable =True)
modis['PM10'].rio.to_raster("./test_transfo/test.tif")
file_output_path = "./test_transfo/test.tif"

with rasterio.open(file_output_path) as raster:
    raster_data = raster.read(1)
    transform = raster.transform 
    
    ... suite procédure 

In [15]:

def process_raster_multibands(file_path, gdf):
    # Open the NetCDF file with rioxarray
    ds = rxr.open_rasterio(file_path, masked=True)

    # List of variables to process
    variables = ['PM10', 'NO2', 'PM25']

    for var_name in variables:
        file_name = os.path.basename(file_path)
        output_raster_file = f"./temp_rasters/{file_name}_{var_name}.tif"
        # Save each variable as a raster
        ds[var_name].rio.to_raster(output_raster_file)

        # Open the saved raster file with rasterio
        with rasterio.open(output_raster_file) as raster:
            raster_data = raster.read(1)
            transform = raster.transform

            def get_raster_value(point):
                row, col = rasterio.transform.rowcol(transform, point.x, point.y)
                return raster_data[row, col]

            # Construct column name based on variable and date
            raster_name = os.path.basename(file_path)
            poll_date = raster_name.split('_')[-1].split('.')[0]
            col_name = f"{var_name}_{poll_date}"

            # Apply function to each point in the GeoDataFrame
            gdf[col_name] = gdf['geometry'].apply(get_raster_value)
    

def process_raster(file_path, gdf):
    with rasterio.open(file_path) as raster:
        raster_data = raster.read(1)  # Charger les données raster
        transform = raster.transform  # Sauvegarder la transformation pour la localisation des points
        
        def get_raster_value(point):
            row, col = rasterio.transform.rowcol(transform, point.x, point.y)
            return raster_data[row, col]
        
        # Extraire le nom du polluant et la date du nom du fichier
        rasterName = os.path.basename(file_path)
        pollName = rasterName.split('_')[0]
        pollDate = rasterName.split('_')[-1].split('.')[0]
        col_name = f"{pollName}_{pollDate}"
        
        # Appliquer la fonction à chaque point du GeoDataFrame
        gdf[col_name] = gdf['geometry'].apply(get_raster_value)


In [9]:
process_raster('../../data/airparif/pollution_chronique/test/horair_IDF_20170106.nc', gdf)

In [23]:

# Measure start time
start_time = time.time()

# Directory paths and initializations
root_dir = '../../data/airparif/pollution_chronique/test/'
gdf_patients = gdf.copy()
files_not_found = []

# Create a directory for temporary raster files
os.makedirs("./temp_rasters", exist_ok=True)

# Process each file
for dirpath, dirnames, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.nc'):  # Ensure only necessary files are processed
            file_path = os.path.join(dirpath, file)
            try:
                if file.split('_')[0] =='horair':
                     process_raster_multibands(file_path, gdf_patients)               
                else :
                    process_raster(file_path, gdf_patients)

            except Exception as e:
                files_not_found.append(file_path)
    print(f"Completed processing for directory: {dirpath}")

# Save the processed GeoDataFrame
# gdf_patients.to_file("../../data/airparif/output/patients_output_pollution.shp", driver='ESRI Shapefile')

# Measure end time
end_time = time.time()
print(f"Temps écoulé : {end_time - start_time} sec")


Completed processing for directory: ['.ipynb_checkpoints', 'pollution', 'pollution_chronique']
Completed processing for directory: []
Completed processing for directory: []
Completed processing for directory: ['.ipynb_checkpoints']
Completed processing for directory: []
Temps écoulé : 26.771878004074097 sec


In [ ]:
file_path = 'O3_IDF_20170105.nc'

In [25]:
gdf_patients.columns

Index(['pseudo_provisoire', 'geometry', 'PM10_20170106', 'NO2_20170106',
       'PM25_20170106', 'NO2_20170104', 'PM10_20170119', 'O3_20170217',
       'O3_20170105', 'PM10_20191229', 'NO2_20191229', 'PM25_20191229'],
      dtype='object')